In [ ]:
import arviz as az
import matplotlib.pyplot as plt
from pathlib import Path
import math
import numpy as np
import os
import pandas as pd

In [ ]:
def select_core_params(infdata: object) -> List[str]:
    """
    Extracts group-level core parameters (drift rate 'v' and decision boundary 'a') 
    from the posterior distributions, filtering out subject-level variance and 
    irrelevant metrics.

    Args:
        infdata (object): An ArviZ InferenceData object containing the posterior traces.

    Returns:
        List[str]: A list of parameter names representing intercepts, main effects, 
        and interactions specifically for 'v' and 'a'.
    """
    all_params = list(infdata.posterior.data_vars.keys())
    selected = []

    for p in all_params:
        # Exclude subject-specific parameters and standard deviation metrics
        if "_subj" in p or "std" in p:
            continue
        
        # Restrict selection to drift rate (v) and decision boundary (a) parameters
        if not (p.startswith("v_") or p.startswith("a_")):
            continue
        
        # Retain model intercepts
        if p.endswith("Intercept"):
            selected.append(p)
            continue
        
        # Retain main effects for judgment_condition
        if "judgment_condition" in p and ":" not in p:
            selected.append(p)
            continue

        # Retain main effects for pair_type
        if "pair_type" in p and ":" not in p:
            selected.append(p)
            continue
        
        # Retain interaction effects
        if ":" in p:
            selected.append(p)
            continue

    return selected


def simplify_labels(params: List[str]) -> Dict[str, str]:
    """
    Converts raw patsy-style parameter names into clean, human-readable labels 
    suitable for academic plots and manuscript tables.

    Args:
        params (List[str]): A list of raw parameter string names extracted from the model.

    Returns:
        Dict[str, str]: A dictionary mapping the original parameter names to their 
        simplified, publication-ready labels.
    """
    labels = {}

    for p in params:
        label = p
        
        # Format base cognitive parameters
        label = label.replace("v_", "v: ")
        label = label.replace("a_", "a: ")
        
        # Simplify categorical treatment contrasts
        label = label.replace("C(pair_type, Treatment('between'))", "Pair")
        label = label.replace("C(judgment_condition, Treatment('categorical'))", "judgment")
        
        # Clean up target level formatting (e.g., converting '[T.level]' to '(level)')
        label = label.replace("[T.", "(").replace("]", ")")
        
        # Standardize interaction symbols for academic formatting
        label = label.replace(":", " × ")
        
        labels[p] = label

    return labels

In [ ]:
# ==============================================================================
# Module 1: Trace Plotting
# Description: Visualizes the sampling process of Markov Chain Monte Carlo (MCMC) chains.
# Function: Inspects parameter convergence and evaluates chain mixing quality.
# ==============================================================================
def plot_trace_custom(infdata, key_parameters, param_labels, save_path):
    """
    Generates and saves custom trace plots for visual inspection of MCMC chains.
    
    Args:
        infdata (object): An ArviZ InferenceData object containing posterior traces.
        key_parameters (List[str]): List of parameter names to plot.
        param_labels (Dict[str, str]): Mapping dictionary from parameter names to 
                                       human-readable labels.
        save_path (Path or str): Destination path for the generated plot.
    """
    chain_colors = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd"]
    chain_labels = [f"Chain {i+1}" for i in range(4)]

    n_params = len(key_parameters)
    n_cols = 2
    n_rows = math.ceil(n_params / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 3 * n_rows))
    axes = np.array(axes).reshape(-1)

    for idx, param in enumerate(key_parameters):
        ax = axes[idx]
        data = infdata.posterior[param]
        n_chains = data.sizes["chain"]
        n_draws = data.sizes["draw"]

        for chain_idx in range(n_chains):
            ax.plot(
                np.arange(n_draws),
                data.sel(chain=chain_idx).values,
                color=chain_colors[chain_idx % len(chain_colors)],
                lw=0.8
            )

        mean_val = data.mean().values
        ax.axhline(mean_val, linestyle="--", linewidth=1)
        ax.set_title(param_labels.get(param, param), fontsize=10)

    # Remove surplus subplots if n_params is odd
    for j in range(idx + 1, len(axes)):
        fig.delaxes(axes[j])

    handles = [
        plt.Line2D([0], [0], color=chain_colors[i], lw=2, label=chain_labels[i])
        for i in range(len(chain_labels))
    ]
    fig.legend(handles=handles, loc="upper center", ncol=4, frameon=False)
    plt.tight_layout(rect=[0, 0, 1, 0.95])
    fig.savefig(save_path, dpi=300)
    plt.close()
    print(f"Trace plot successfully saved to: {save_path}")


# ==============================================================================
# Module 2: Model Diagnostics
# Description: Automatically computes statistical metrics to evaluate model fit quality.
# Function: Screens R-hat (convergence diagnostic) and ESS (Effective Sample Size), 
#           and exports anomalous parameters to CSV for methodological reporting.
# ==============================================================================
def run_diagnostics(infdata, save_dir, model_name):
    """
    Executes convergence diagnostics and saves the summary statistics.
    
    Args:
        infdata (object): An ArviZ InferenceData object.
        save_dir (Path): Output directory for the diagnostic reports.
        model_name (str): Identifier for the current model.
        
    Returns:
        pd.DataFrame: A comprehensive summary dataframe of all diagnostic metrics.
    """
    summary_df = az.summary(
        infdata,
        var_names=list(infdata.posterior.data_vars.keys()),
        kind="diagnostics",
        round_to=4
    )

    # Screen R-hat: Values > 1.01 typically indicate a lack of convergence
    bad_rhat = summary_df[summary_df['r_hat'] > 1.01]
    print(f"Diagnostic Alert: {len(bad_rhat)} parameters with R-hat > 1.01")
    if not bad_rhat.empty:
        bad_rhat[['r_hat']].to_csv(save_dir / f"{model_name}_bad_rhat.csv")

    # Screen ESS: Values < 400 suggest inefficient sampling and high autocorrelation
    bad_ess = summary_df[(summary_df['ess_bulk'] < 400) | (summary_df['ess_tail'] < 400)]
    print(f"Diagnostic Alert: {len(bad_ess)} parameters with ESS < 400")
    if not bad_ess.empty:
        bad_ess[['ess_bulk', 'ess_tail']].to_csv(save_dir / f"{model_name}_bad_ess.csv")

    summary_df.to_csv(save_dir / f"{model_name}_diagnostics.csv")
    print(f"Diagnostics completed and saved for model: {model_name}")
    return summary_df


# ==============================================================================
# Module 3: Parameter Computation
# Description: Core computational logic for extracting posterior parameters 
#              structured via linear regression design matrices.
# Function: Reconstructs condition-specific parameter distributions by aggregating 
#           intercepts, main effects, and interaction terms. Computes the posterior 
#           mean and Bayesian standard error (SE).
# ==============================================================================
def compute_param(trace_df, prefix, pair, judgment, subj=None):
    """
    Extracts and aggregates parameters for specific experimental conditions from 
    posterior traces. Includes a robust fallback mechanism to ignore undefined columns.
    
    Args:
        trace_df (pd.DataFrame): DataFrame containing the flattened posterior traces.
        prefix (str): Parameter prefix (e.g., 'a_' for boundary, 'v_' for drift rate).
        pair (str): The specific level of the 'pair_type' condition.
        judgment (str): The specific level of the 'judgment_condition' condition.
        subj (str, optional): Subject identifier for extracting subject-level effects.
        
    Returns:
        Tuple[float, float]: The computed posterior mean and Bayesian standard error.
    """
    potential_cols = []
    
    # 1. Base component: Model Intercept
    potential_cols.append(f"{prefix}Intercept")

    # 2. Main effect: pair_type
    if pair != "between":
        potential_cols.append(f"{prefix}C(pair_type, Treatment('between'))[T.{pair}]")

    # 3. Main effect: judgment_condition
    if judgment != "categorical":
        potential_cols.append(f"{prefix}C(judgment_condition, Treatment('categorical'))[T.{judgment}]")

    # 4. Interaction effect
    if pair != "between" and judgment != "categorical":
        potential_cols.append(f"{prefix}C(judgment_condition, Treatment('categorical'))[T.{judgment}]:C(pair_type, Treatment('between'))[T.{pair}]")

    # 5. Subject-specific random effects
    if subj is not None:
        subj_cols = [f"{c}_subj.{subj}" for c in potential_cols]
        potential_cols.extend(subj_cols)

    # Robustness Check: Retain only those columns explicitly present in the design matrix
    actual_cols = [c for c in potential_cols if c in trace_df.columns]

    # Fallback: Return zero if no columns match (should only occur with incorrect prefixes)
    if not actual_cols:
        return 0.0, 0.0

    # Aggregate traces row-wise to reconstruct the parameter distribution for this condition
    samples = trace_df[actual_cols].sum(axis=1)
    
    # Note: In Bayesian posteriors, the standard deviation of the samples represents 
    # the parameter's uncertainty, equivalent to the Bayesian Standard Error (SE).
    post_mean = samples.mean()
    post_se = samples.std() 
    
    return post_mean, post_se


# ==============================================================================
# Module 4: Condition Plotting & CSV Export
# Description: Visualizes condition-specific parameters and exports aggregated 
#              data to CSV for downstream statistical analyses (e.g., in R).
# Function: Generates line plots with dodge-adjusted Standard Error (SE) bars.
# ==============================================================================
def plot_param(df, mean_col, err_col, title, save_path):
    """
    Generates a mean-and-error plot for experimental conditions and automatically 
    saves the underlying data to an identically named CSV file.
    
    Args:
        df (pd.DataFrame): Dataset containing the plotting data.
        mean_col (str): Column name representing the posterior mean.
        err_col (str): Column name representing the standard error.
        title (str): Title of the plot.
        save_path (str or Path): Target file path for the plot. The CSV is saved alongside it.
    """
    # ---------------------------------------------------------
    # Data Export Logic
    # ---------------------------------------------------------
    # Deriving CSV path dynamically from the image path
    csv_save_path = str(save_path).rsplit('.', 1)[0] + ".csv"
    
    # Ensure target directory exists
    dir_name = os.path.dirname(csv_save_path)
    if dir_name:
        os.makedirs(dir_name, exist_ok=True)
    
    # Export using utf-8-sig to ensure universal cross-platform compatibility
    df.to_csv(csv_save_path, index=False, encoding='utf-8-sig')
    print(f"Underlying plot data exported to CSV: {csv_save_path}")

    # ---------------------------------------------------------
    # Plotting Logic
    # ---------------------------------------------------------
    plt.figure(figsize=(5, 4))
    
    # Implement a slight x-axis dodge offset to prevent overlapping error bars
    offsets = {"between": -0.05, "within": 0.05}
    
    for r in ["between", "within"]:
        sub = df[df["pair_type"] == r].copy()
        
        # Convert categorical x-axis to numeric array for manual dodge offset
        x_numeric = np.arange(len(sub["judgment_condition"]))
        
        plt.errorbar(
            x_numeric + offsets.get(r, 0), 
            sub[mean_col],
            yerr=sub[err_col], 
            marker="o", 
            capsize=4, 
            elinewidth=1.5,
            markersize=6,
            label=r
        )
        
        # Align x-axis tick labels accurately
        plt.xticks(x_numeric, sub["judgment_condition"])

    plt.title(title, pad=15)
    plt.xlabel("Judgment Condition")
    plt.ylabel(f"{mean_col} (\u00B1 SE)")  # Standardized unicode for ± 
    plt.legend(title="Pair Type")
    plt.grid(alpha=0.3, linestyle="--")
    plt.tight_layout()
    
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Condition plot successfully saved to: {save_path}")

In [ ]:
# ==============================================================================
# Master Execution Pipeline
# ==============================================================================

def run_full_analysis(
    nc_file: str, 
    model_name: str, 
    params_to_check: List[str] = ["v", "a", "t", "z"]
) -> None:
    """
    Executes a comprehensive post-hoc analysis pipeline for a fitted HDDM model.
    Reads posterior traces, generates diagnostics, computes condition-level 
    and subject-level parameters, and automatically exports plots and CSVs.
    
    Args:
        nc_file (str or Path): Path to the NetCDF file containing the posterior traces.
        model_name (str): Identifier for the model being analyzed.
        params_to_check (List[str], optional): List of core cognitive parameters to 
            evaluate automatically. Defaults to ["v", "a", "t", "z"] (drift rate, 
            decision boundary, non-decision time, and starting point).
    """
    save_dir = Path(f"output/result/formal_model/{model_name}")
    save_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n[INFO] Initiating comprehensive analysis for model: {model_name}")

    # Load posterior data
    infdata = az.from_netcdf(nc_file)

    # Extract and simplify parameter labels for plotting
    key_parameters = select_core_params(infdata)
    param_labels = simplify_labels(key_parameters)

    print("\n[INFO] Selected parameters for evaluation:")
    for p in key_parameters:
        print(f"  - {p}")

    # Generate MCMC trace plots
    plot_trace_custom(
        infdata,
        key_parameters,
        param_labels,
        save_dir / f"{model_name}_trace.png"
    )

    # Run statistical convergence diagnostics (R-hat, ESS)
    run_diagnostics(infdata, save_dir, model_name)

    trace_df = infdata.posterior.to_dataframe().reset_index()
    pair_types = ["between", "within"]
    judgment_conditions = ["categorical", "continuous"]

    # ==============================================================================
    # Core Implementation: Dynamic Parameter Detection
    # ==============================================================================
    # Verify the presence of intercept terms in the posterior traces to confirm 
    # which parameters were actually estimated in the regression model.
    valid_params = [p for p in params_to_check if f"{p}_Intercept" in trace_df.columns]
    
    if not valid_params:
        print("[WARNING] No parameters with an '_Intercept' suffix detected. "
              "Please ensure the model was formulated using HDDMRegressor.")
    else:
        print(f"[INFO] Dynamically detected parameters for condition-level extraction: {valid_params}")

    # ==============================================================================
    # 1. Group-Level Parameter Computation
    # ==============================================================================
    records = []
    for r in judgment_conditions:
        for n in pair_types:
            # Initialize base dictionary for the current experimental condition
            row_data = {"judgment_condition": r, "pair_type": n}
            
            # Iterate across all valid cognitive parameters to compute condition estimates
            for p in valid_params:
                p_mean, p_std = compute_param(trace_df, f"{p}_", n, r)
                row_data[f"{p}_mean"] = p_mean
                row_data[f"{p}_std"] = p_std
                
            records.append(row_data)

    df_group = pd.DataFrame(records)
    df_group["pair_type"] = pd.Categorical(df_group["pair_type"], ["between", "within"], ordered=True)
    df_group = df_group.sort_values(["judgment_condition", "pair_type"])

    # ==============================================================================
    # 2. Automated Parameter Visualization
    # ==============================================================================
    # Predefined mapping for formal plot titles
    title_map = {
        "v": "Drift rate (v)", 
        "a": "Boundary (a)", 
        "t": "Non-decision time (t)", 
        "z": "Starting point (z)"
    }
    
    for p in valid_params:
        title = title_map.get(p, f"Parameter ({p})")
        save_path = save_dir / f"{model_name}_{p}_plot.png"
        plot_param(df_group, f"{p}_mean", f"{p}_std", title, save_path)

    # ==============================================================================
    # 3. Subject-Level Parameter Extraction
    # ==============================================================================
    # Isolate subject indices dynamically from the trace dataframe
    subjs = sorted({int(col.split(".")[-1]) for col in trace_df.columns if "_subj." in col})

    subj_records = []
    for subj in subjs:
        for r in judgment_conditions:
            for n in pair_types:
                row_data = {"subj": subj, "judgment_condition": r, "pair_type": n}
                
                # Compute individual estimates for each valid cognitive parameter
                for p in valid_params:
                    p_mean, _ = compute_param(trace_df, f"{p}_", n, r, subj)
                    row_data[f"{p}_mean"] = p_mean
                    
                subj_records.append(row_data)

    df_subj = pd.DataFrame(subj_records)
    df_subj.to_csv(save_dir / f"{model_name}_subj.csv", index=False)

    print(f"[SUCCESS] Analysis pipeline completed for model: {model_name}")

In [ ]:
from pathlib import Path

# ==============================================================================
# Batch Execution Script
# Description: Automates the execution of the full analysis pipeline across 
#              all successfully fitted models in the designated directory.
# ==============================================================================

if __name__ == "__main__":
    # 1. Define the primary base directory containing the fitted models
    base_dir = Path("../../reference_outputs/hddm/formal_models")
    
    # 2. Recursively locate all compiled NetCDF (.nc) model files.
    # Assumes a directory architecture of: base_dir / model_name / model_name_fitted.nc
    nc_files = list(base_dir.rglob("*_fitted.nc"))
    
    if not nc_files:
        print(f"[WARNING] No .nc files located in {base_dir}. Please verify the directory path.")
    else:
        print(f"[INFO] Successfully scanned {len(nc_files)} model files. Initiating batch diagnostic pipeline...")
        
        # 3. Iterate through all detected model files
        for nc_file in nc_files:
            # Dynamically extract the model name from its parent directory
            # (e.g., extracting 'M9_v_interaction_a_train')
            model_name = nc_file.parent.name 
            
            print(f"\n{'='*60}")
            print(f"[INFO] Processing model: {model_name}")
            print(f"{'='*60}")
            
            # 4. Implement fault-tolerance to ensure continuous execution across the batch
            try:
                run_full_analysis(
                    nc_file=str(nc_file),  # Cast Path object to string for compatibility
                    model_name=model_name
                )
            except Exception as e:
                # Log the exception and proceed to the next model without terminating the pipeline
                print(f"\n[ERROR] Diagnostic failure for model {model_name}. Skipping to the next iteration.")
                print(f"[ERROR] Exception trace: {e}")
                continue
                
        print("\n[SUCCESS] Batch diagnostic pipeline completed successfully.")